# Airport Runway FOD Detection — YOLOv8n Baseline Training

This Colab notebook trains the reportable original-image baseline using the frozen leakage-resistant split. It reproduces the local conversion and training configuration; it does not create a new random split.

Before running, select **Runtime → Change runtime type → T4 GPU** (or another available GPU). Upload `FODPascalVOCFormat-V.2.1.zip` to Google Drive.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Confirm that Colab assigned a CUDA GPU.
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable a GPU from Runtime > Change runtime type before continuing.'
print('CUDA device:', torch.cuda.get_device_name(0))

Sat Nov  7 17:56:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Mount Google Drive and configure paths

Change `DRIVE_DATASET_ZIP` if the uploaded ZIP is stored in another Drive folder. Training outputs will be copied to `DRIVE_RESULTS_DIR`.

In [5]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

DRIVE_DATASET_ZIP = Path('/content/drive/MyDrive/Machine Vision/FODPascalVOCFormat-V.2.1.zip')
DRIVE_RESULTS_DIR = Path('/content/drive/MyDrive/Machine Vision/FOD Baseline Results')
REPOSITORY_DIR = Path('/content/airport-runway-fod-detection')
assert DRIVE_DATASET_ZIP.exists(), f'Dataset ZIP not found: {DRIVE_DATASET_ZIP}'
print('Dataset ZIP:', DRIVE_DATASET_ZIP)
print('Results destination:', DRIVE_RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset ZIP: /content/drive/MyDrive/Machine Vision/FODPascalVOCFormat-V.2.1.zip
Results destination: /content/drive/MyDrive/Machine Vision/FOD Baseline Results


## 2. Obtain the version-controlled project

The exact `baseline-model-training` branch is used so the frozen split, converter and training settings match the audited project.

In [10]:
import shutil
import subprocess
from google.colab import userdata

if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)

# If this is a private repository, add your token to Colab's Secrets (left sidebar, key icon) as 'GITHUB_TOKEN'
try:
    token = userdata.get('GITHUB_TOKEN')
    repo_url = f'https://{token}@github.com/neelia26/airport-runway-fod-detection.git'
    print("Attempting to clone using stored GITHUB_TOKEN...")
except Exception:
    # Fallback to public URL if no token is saved
    repo_url = 'https://github.com/neelia26/airport-runway-fod-detection.git'
    print("No GITHUB_TOKEN found. Attempting public clone...")

subprocess.run([
    'git', 'clone', '--branch', 'baseline-model-training', '--single-branch',
    repo_url, str(REPOSITORY_DIR),
], check=True)

%cd /content/airport-runway-fod-detection
!git log -1 --oneline

No GITHUB_TOKEN found. Attempting public clone...
/content/airport-runway-fod-detection
f50d712 (HEAD -> baseline-model-training, origin/baseline-model-training) Add Colab baseline training notebook


## 3. Install the training environment

In [11]:
%pip install -q -e '.[training]'
!python scripts/check_training_environment.py

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for airport-runway-fod-detection (pyproject.toml) ... done
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Python: 3.13.16
PyTorch: 2.14.1+cu130
Ultralytics: 8.4.205
MPS built: False
MPS available: False
CUDA available: True
CUDA device: Tesla T4


## 4. Extract the untouched Pascal VOC source dataset

Extraction occurs in Colab's temporary storage. The ZIP in Drive is read but not modified.

In [12]:
import zipfile

dataset_parent = REPOSITORY_DIR / 'FOD-A Dataset'
dataset_parent.mkdir(parents=True, exist_ok=True)
expected_voc_root = dataset_parent / 'FODPascalVOCFormat-V.2.1' / 'VOC2007'
if not expected_voc_root.exists():
    with zipfile.ZipFile(DRIVE_DATASET_ZIP) as archive:
        archive.extractall(dataset_parent)
assert expected_voc_root.exists(), f'Unexpected ZIP structure: {expected_voc_root} not found'
print('Pascal VOC dataset:', expected_voc_root)

Pascal VOC dataset: /content/airport-runway-fod-detection/FOD-A Dataset/FODPascalVOCFormat-V.2.1/VOC2007


## 5. Reproduce the verified single-class YOLO conversion

This uses the committed sequence-aware train, validation and test manifests. All 31 source categories are mapped to class `0: FOD`.

In [13]:
!MPLBACKEND=Agg python scripts/convert_voc_to_yolo.py
!python scripts/verify_yolo_conversion.py

Converted 5,000 images
Converted 10,000 images
Converted 15,000 images
Converted 20,000 images
Converted 25,000 images
Converted 30,000 images
Converted 33,793 images
Converted 34,472 objects to class 0: FOD
Maximum round-trip error: 0.00000150 pixels
Generated dataset: data/yolo
PASS: all_manifest_images_present
PASS: one_label_file_per_image
PASS: all_objects_preserved
PASS: all_labels_are_single_class_fod
PASS: 33,793 image-label pairs
PASS: 34,472 single-class boxes


## 6. Optional pipeline smoke test

This one-epoch subset run is only a technical check. Its metrics must not be reported as research results.

In [ ]:
!python scripts/train_baseline.py --smoke-test --device cuda

## 7. Train the reportable baseline

The command below trains YOLOv8n for at most 30 epochs with early stopping. Do not change the split, image size, batch size or seed if this run will be compared with later preprocessing variants.

In [ ]:
!python scripts/train_baseline.py --device cuda

## 8. Preserve the complete run in Google Drive

Colab storage is temporary. This cell copies checkpoints, metrics and figures to Drive after training.

In [ ]:
from datetime import datetime

source_run = REPOSITORY_DIR / 'runs' / 'baseline' / 'yolov8n_original'
assert source_run.exists(), f'Completed run not found: {source_run}'
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
destination = DRIVE_RESULTS_DIR / f'yolov8n_original_{timestamp}'
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source_run, destination)
print('Saved complete baseline run to:', destination)

## Next step

Do not interpret the final metrics in isolation. The saved `best.pt` checkpoint must next be evaluated on the frozen test partition by object size, lighting and weather. Gaussian and median variants should only be trained after the original-image baseline has been verified.